# 05 - Demo (Gradio, Local, Pretrained Models Only)

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import joblib
import gradio as gr

from src.features import PhysicsFeatures
from src.oracle import oracle_rule_breakdown

NUMERIC_COLS = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load pretrained models and reference data

In [2]:
MODELS_DIR = REPO_ROOT / "models"

MODELS = {
    ("A", "logreg"): joblib.load(MODELS_DIR / "logreg_A.joblib"),
    ("A", "tree"): joblib.load(MODELS_DIR / "tree_A.joblib"),
    ("A", "knn"): joblib.load(MODELS_DIR / "knn_A.joblib"),
    ("B", "logreg"): joblib.load(MODELS_DIR / "logreg_B.joblib"),
    ("B", "tree"): joblib.load(MODELS_DIR / "tree_B.joblib"),
    ("B", "knn"): joblib.load(MODELS_DIR / "knn_B.joblib"),
    ("C", "logreg"): joblib.load(MODELS_DIR / "logreg_C.joblib"),
    ("C", "tree"): joblib.load(MODELS_DIR / "tree_C.joblib"),
    ("C", "knn"): joblib.load(MODELS_DIR / "knn_C.joblib"),
}

splits = joblib.load(REPO_ROOT / "results" / "feature_splits.joblib")
poly_transformer = splits["poly_transformer"]
X_train_A = splits["X_train_A"]

TRAIN_RANGES = X_train_A[NUMERIC_COLS].agg(["min", "max"])
TRAIN_RANGES


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
min,295.3,305.7,1168,3.8,0
max,304.5,313.8,2886,76.6,253


## Feature builders for each scenario, from raw user input

In [3]:
physics = PhysicsFeatures(add_osf_margin=True)

def build_scenario_A(raw_row):
    return raw_row.copy()

def build_scenario_B(raw_row):
    poly_cols = poly_transformer.transform(raw_row[NUMERIC_COLS])
    poly_df = pd.DataFrame(
        poly_cols,
        columns=poly_transformer.get_feature_names_out(NUMERIC_COLS),
        index=raw_row.index,
    )
    poly_only = poly_df.drop(columns=NUMERIC_COLS)
    return pd.concat([raw_row.copy(), poly_only], axis=1)

def build_scenario_C(raw_row):
    return physics.transform(raw_row)

SCENARIO_BUILDERS = {"A": build_scenario_A, "B": build_scenario_B, "C": build_scenario_C}


## Input range check

In [4]:
def range_warnings(raw_row):
    warnings = []
    for col in NUMERIC_COLS:
        lo, hi = TRAIN_RANGES.loc["min", col], TRAIN_RANGES.loc["max", col]
        val = raw_row[col].iloc[0]
        if val < lo or val > hi:
            warnings.append(f"{col} = {val} is outside the training range [{lo}, {hi}]")
    process_temp = raw_row["Process temperature [K]"].iloc[0]
    air_temp = raw_row["Air temperature [K]"].iloc[0]
    if process_temp < air_temp:
        warnings.append("Process temperature is below air temperature")
    return warnings


## Prediction function

In [5]:
def predict(product_type, air_temp, process_temp, rpm, torque, tool_wear, scenario, model_key):
    raw_row = pd.DataFrame([{
        "Type": product_type,
        "Air temperature [K]": air_temp,
        "Process temperature [K]": process_temp,
        "Rotational speed [rpm]": rpm,
        "Torque [Nm]": torque,
        "Tool wear [min]": tool_wear,
    }])

    warnings = range_warnings(raw_row)
    features = SCENARIO_BUILDERS[scenario](raw_row)
    model = MODELS[(scenario, model_key)]

    pred = int(model.predict(features)[0])
    score = float(model.predict_proba(features)[0, 1])

    rules = oracle_rule_breakdown(raw_row).iloc[0]
    triggered = [name for name, hit in rules.items() if hit]

    label = "FAILURE" if pred == 1 else "NO FAILURE"
    rule_text = ", ".join(triggered) if triggered else "none"
    warning_text = "; ".join(warnings) if warnings else "none"

    result = (
        f"Prediction ({model_key}, scenario {scenario}): {label}\n"
        f"Failure score: {score:.3f}\n"
        f"Reference rule check (not the model): {rule_text}\n"
        f"Input warnings: {warning_text}"
    )
    return result


## Examples from data/demo_input.csv

In [6]:
demo_df = pd.read_csv(REPO_ROOT / "data" / "demo_input.csv")

examples = []
for _, row in demo_df.iterrows():
    examples.append([
        row["Type"], row["Air temperature [K]"], row["Process temperature [K]"],
        row["Rotational speed [rpm]"], row["Torque [Nm]"], row["Tool wear [min]"],
        "C", "tree",
    ])

demo_df


,case,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,healthy,M,298.1,308.6,1551,42.8,0,0
1,HDF,M,300.8,309.4,1342,62.4,113,1
2,PWF,L,298.9,309.1,2861,4.6,143,1
3,OSF,L,298.9,309.0,1410,65.7,191,1


## Gradio interface

In [7]:
interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Dropdown(["L", "M", "H"], value="M", label="Type"),
        gr.Number(value=300.0, label="Air temperature [K]"),
        gr.Number(value=310.0, label="Process temperature [K]"),
        gr.Number(value=1500, label="Rotational speed [rpm]"),
        gr.Number(value=40.0, label="Torque [Nm]"),
        gr.Number(value=100, label="Tool wear [min]"),
        gr.Dropdown(["A", "B", "C"], value="C", label="Scenario"),
        gr.Dropdown(["logreg", "tree", "knn"], value="tree", label="Model"),
    ],
    outputs=gr.Textbox(label="Result", lines=6),
    examples=examples,
    title="AI4I 2020 Machine Failure Demo",
)

interface.launch(share=False, prevent_thread_lock=True)


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:99: UserWarning: unable to parse version details from package URL.
  warnings.warn("unable to parse version details from package URL.")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Fallback: predictions without Gradio

In [8]:
for _, row in demo_df.iterrows():
    text = predict(
        row["Type"], row["Air temperature [K]"], row["Process temperature [K]"],
        row["Rotational speed [rpm]"], row["Torque [Nm]"], row["Tool wear [min]"],
        "C", "tree",
    )
    print(row["case"])
    print(text)
    print()


healthy
Prediction (tree, scenario C): NO FAILURE
Failure score: 0.027
Reference rule check (not the model): none
Input warnings: none

HDF
Prediction (tree, scenario C): FAILURE
Failure score: 0.977
Reference rule check (not the model): HDF
Input warnings: none

PWF
Prediction (tree, scenario C): FAILURE
Failure score: 1.000
Reference rule check (not the model): PWF
Input warnings: none

OSF
Prediction (tree, scenario C): FAILURE
Failure score: 1.000
Reference rule check (not the model): PWF, OSF
Input warnings: none

